# Aula 02 — Modelo do Espaço Vetorial: vetores, TF-IDF e similaridade do cosseno
**Projeto Integrador III — Motor de Busca**
*Kernel: R | Módulos 1 a 8 e Teste Final*

In [1]:
# 1. Setup inicial do motor02
source("https://raw.githubusercontent.com/fractalarea/pi3-motor-de-busca/main/motor/motor02.R")
docs <- docs_aula()
cfg  <- cfg_aula()
ix   <- montar(docs, cfg)
estado()

[1] "MOTOR_VERSAO: motor02 (Aula 01 -> Aula 02)"
docs: 8 documentos carregados. ix: lista com 7 elementos (vocab=45, N=8, tf 45x8, w 45x8).


In [2]:
# 2. Funcao de similaridade do cosseno
cosseno <- function(a, b) {
  den <- sqrt(sum(a^2)) * sqrt(sum(b^2))
  if (den == 0) return(0)
  sum(a * b) / den
}

cosseno(c(1, 1), c(2, 1))
cosseno(c(1, 1), c(4, 2))  # dobrar tamanho nao altera cosseno
cosseno(c(0, 0), c(2, 1))  # vetor nulo devolve 0 (sem NaN)
cosseno(c(1, 2), c(2, 1))  # Checkpoint 4

[1] 0.9486833
[1] 0.9486833
[1] 0
[1] 0.8


In [3]:
# 3. Normalizacao por coluna (vetores unitarios)
norm_cols <- function(m) {
  nrm <- sqrt(colSums(m^2))
  nrm[nrm == 0] <- 1
  sweep(m, 2, nrm, "/")
}

wn <- norm_cols(ix$w)
round(colSums(wn^2), 2)
sum(wn[, "d2"] * wn[, "d3"])  # Checkpoint 5

d1 d2 d3 d4 d5 d6 d7 d8 
 1  1  1  1  1  1  1  1 
[1] 0.09170673


In [4]:
# 4. Vetor da consulta e ranking
vetor_consulta <- function(termos, vocab, idf) {
  q <- as.integer(table(factor(termos, levels = vocab)))
  q * idf
}

qw <- vetor_consulta(tokenizar("modelo de recuperacao"), ix$vocab, ix$idf_tfidf)
round(qw[qw > 0], 2)

ranking_cosseno <- function(consulta, ix) {
  qw <- vetor_consulta(tokenizar(consulta), ix$vocab, ix$idf_tfidf)
  s  <- apply(ix$w, 2, function(d) cosseno(qw, d))
  sort(s, decreasing = TRUE)
}

round(ranking_cosseno("modelo de recuperacao", ix), 3)

         de      modelo recuperacao 
       0.47        1.39        1.39 
   d1    d3    d4    d2    d6    d8    d5    d7 
0.254 0.233 0.215 0.208 0.025 0.023 0.000 0.000 
